# Top-1 Cosine Clustered FedAvg with Non-IID (Dirichlet) data

This notebook keeps the FedAvg training, checkpoint, resume, and metrics flow. After local training, client updates are clustered with elbow KMeans. Each cluster computes a sample-weighted centroid, selects the client with the highest cosine similarity to that centroid, and the server performs sample-weighted FedAvg over the selected representatives.

In [ ]:
import csv
import hashlib
import json
import os
import random
import tarfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, models

try:
    get_ipython().run_line_magic('load_ext', 'cuml.accel')
    print('cuML acceleration enabled for sklearn KMeans.')
except Exception as error:
    print(f'cuML acceleration unavailable; using CPU sklearn KMeans: {error}')
from sklearn.cluster import KMeans

assert torch.cuda.is_available(), 'Enable the T4 GPU in Colab before running this notebook.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print(torch.cuda.get_device_name(0))

In [ ]:
# Main configuration
DIRICHLET_ALPHA = 0.1
MIN_SAMPLES = 100
PARTITION_TYPE = 'noniid_2'  # Use noniid_2_25clients_seed42.pt
MODEL_NAME = 'cnn'  # 'cnn', 'logreg_raw', or 'resnet_linear'
DATASET_NAME = 'mnist'  # 'mnist' or 'cifar10'
NUM_CLIENTS = 25
CLIENTS_PER_ROUND = 25
ROUNDS = 100
LOCAL_EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 0.01
RESNET_SIZE = 112
EMBED_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 4096

# Clustering configuration
K_MAX = 15
CLUSTER_SEED = 44

PARTITION_SEED = 42
MODEL_SEED = 42
BATCH_SEED = 42
SELECTION_SEED = 43

# Run configuration
RUN_ID = 2

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/FLasIRL(new)')
except ImportError:
    print('Not running in Colab; using local directory fallback.')
    PROJECT_ROOT = Path.cwd() / 'FLasIRL(new)' if not Path('/content').exists() else Path('/content/FLasIRL(new)')

if not PROJECT_ROOT.parent.exists():
    PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / DATASET_NAME
PARTITION_DIR = DATA_DIR / 'partitions' / 'Dirichlet'
EXP_DIR = PROJECT_ROOT / 'top1_cosine'
for folder in [DATA_DIR, PARTITION_DIR, EXP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
run_folder_name = str(RUN_ID) if str(RUN_ID).startswith('run') else f'run{RUN_ID}'
RUN_DIR = EXP_DIR / run_folder_name
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'metrics.csv'
CONFIG_PATH = RUN_DIR / 'config.json'
CHECKPOINT_PATH = RUN_DIR / 'checkpoint_last.pt'
CLUSTER_LOG_PATH = RUN_DIR / 'cluster_assignments.json'
print('Run ID:', RUN_ID)
print('Run directory:', RUN_DIR)
print('Project root:', PROJECT_ROOT)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

if DATASET_NAME == 'mnist':
    train_ds = datasets.MNIST(DATA_DIR, train=True, download=True)
    test_ds = datasets.MNIST(DATA_DIR, train=False, download=True)
    train_images = train_ds.data.float().div_(255.0)
    test_images = test_ds.data.float().div_(255.0)
    train_labels = train_ds.targets.long()
    test_labels = test_ds.targets.long()
    DATA_MEAN, DATA_STD = 0.1307, 0.3081
    INPUT_CHANNELS, INPUT_SIZE = 1, 28
else:
    extracted_dir = DATA_DIR / 'cifar-10-batches-py'
    if not extracted_dir.exists():
        archives = sorted(DATA_DIR.glob('*.gz'))
        if archives:
            with tarfile.open(archives[0], 'r:gz') as archive:
                archive.extractall(DATA_DIR)
    train_ds = datasets.CIFAR10(DATA_DIR, train=True, download=True)
    test_ds = datasets.CIFAR10(DATA_DIR, train=False, download=True)
    train_images = torch.from_numpy(train_ds.data).permute(0, 3, 1, 2).float().div_(255.0)
    test_images = torch.from_numpy(test_ds.data).permute(0, 3, 1, 2).float().div_(255.0)
    train_labels = torch.tensor(train_ds.targets, dtype=torch.long)
    test_labels = torch.tensor(test_ds.targets, dtype=torch.long)
    DATA_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
    DATA_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
    INPUT_CHANNELS, INPUT_SIZE = 3, 32

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
if MODEL_NAME == 'cnn':
    train_features = ((train_images - DATA_MEAN) / DATA_STD)
    test_features = ((test_images - DATA_MEAN) / DATA_STD)
    if DATASET_NAME == 'mnist':
        train_features = train_features.unsqueeze(1)
        test_features = test_features.unsqueeze(1)
    train_features = train_features.to(DEVICE)
    test_features = test_features.to(DEVICE)
elif MODEL_NAME == 'logreg_raw':
    train_features = ((train_images - DATA_MEAN) / DATA_STD).reshape(len(train_images), -1).to(DEVICE)
    test_features = ((test_images - DATA_MEAN) / DATA_STD).reshape(len(test_images), -1).to(DEVICE)
else:
    train_features = None
    test_features = None
train_labels_gpu = train_labels.to(DEVICE)
test_labels_gpu = test_labels.to(DEVICE)
print('Train:', train_images.shape, 'Test:', test_images.shape)

In [ ]:
import re

def generate_dirichlet_partition(labels, num_clients=25, alpha=0.5, seed=42, min_samples=10, max_retries=100):
    labels_np = labels.cpu().numpy()
    num_samples = len(labels_np)
    classes = np.unique(labels_np)
    num_classes = len(classes)
    for attempt in range(max_retries):
        rng = np.random.default_rng(seed + attempt)
        client_sample_indices = [[] for _ in range(num_clients)]
        for c in classes:
            idx_c = np.where(labels_np == c)[0]
            rng.shuffle(idx_c)
            counts = rng.multinomial(len(idx_c), rng.dirichlet(np.repeat(alpha, num_clients)))
            start = 0
            for client_id, count in enumerate(counts):
                client_sample_indices[client_id].extend(idx_c[start:start + count])
                start += count
        if min(map(len, client_sample_indices)) >= min_samples:
            client_indices = []
            client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)
            for client_id, indices in enumerate(client_sample_indices):
                indices_arr = np.asarray(indices, dtype=np.int64)
                rng.shuffle(indices_arr)
                t_indices = torch.tensor(indices_arr, dtype=torch.long)
                client_indices.append(t_indices)
                client_class_counts[client_id] = torch.bincount(labels[t_indices], minlength=num_classes)
            assert sum(map(len, client_indices)) == num_samples
            assert len(torch.cat(client_indices).unique()) == num_samples
            return client_indices, client_class_counts
    raise RuntimeError(f'Could not generate Dirichlet partition with alpha={alpha}.')

# Parse PARTITION_TYPE exactly as in partition_dirichlet.ipynb.
def _parse_partition_type(partition_type):
    if partition_type == 'dirichlet':
        return 'dirichlet', None
    match = re.fullmatch(r'noniid_(\d+)', partition_type)
    if match:
        return 'noniid', int(match.group(1))
    raise ValueError(
        f"PARTITION_TYPE không hợp lệ: '{partition_type}'. "
        "Dùng 'dirichlet' hoặc 'noniid_N' (ví dụ: 'noniid_2')."
    )

partition_kind, classes_per_client = _parse_partition_type(PARTITION_TYPE)
if partition_kind == 'dirichlet':
    stem = f'dirichlet_alpha{DIRICHLET_ALPHA}_{NUM_CLIENTS}clients_seed{PARTITION_SEED}'
else:
    stem = f'noniid_{classes_per_client}_{NUM_CLIENTS}clients_seed{PARTITION_SEED}'
partition_path = PARTITION_DIR / f'{stem}.pt'
if not partition_path.exists():
    raise FileNotFoundError(
        f'Partition không tồn tại: {partition_path}. '
        f'Hãy chạy partition_dirichlet.ipynb với PARTITION_TYPE=\"{PARTITION_TYPE}\" trước.'
    )
print(f'Loading {PARTITION_TYPE} partition from {partition_path}...')
partition = torch.load(partition_path, map_location='cpu', weights_only=False)
if partition.get('num_clients', NUM_CLIENTS) != NUM_CLIENTS:
    raise ValueError(
        f'Partition có num_clients={partition.get("num_clients")}, '
        f'nhưng notebook yêu cầu NUM_CLIENTS={NUM_CLIENTS}.'
    )
if partition.get('distribution') not in (None, PARTITION_TYPE):
    raise ValueError(
        f'Partition có distribution={partition.get("distribution")!r}, '
        f'không khớp PARTITION_TYPE={PARTITION_TYPE!r}.'
    )
client_indices = partition['client_indices']
assert len(client_indices) == NUM_CLIENTS
if 'class_counts' in partition:
    client_class_counts = partition['class_counts']
else:
    client_class_counts = torch.zeros(NUM_CLIENTS, 10, dtype=torch.long)
    for client_id, indices in enumerate(client_indices):
        client_class_counts[client_id] = torch.bincount(train_labels[indices], minlength=10)

assert sum(len(x) for x in client_indices) == len(train_labels)
assert len(torch.cat(client_indices).unique()) == len(train_labels)
client_sizes = np.array([len(x) for x in client_indices])
print(f'Client sizes: min={client_sizes.min()}, max={client_sizes.max()}, mean={client_sizes.mean():.1f}, std={client_sizes.std():.1f}')

plt.figure(figsize=(14, 6))
bottom = np.zeros(NUM_CLIENTS)
for cls, color in enumerate(plt.cm.tab10(np.linspace(0, 1, 10))):
    values = client_class_counts[:, cls].cpu().numpy()
    plt.bar(np.arange(NUM_CLIENTS), values, bottom=bottom, label=f'Class {cls}', color=color, edgecolor='black', linewidth=0.5)
    bottom += values
plt.xlabel('Client ID')
plt.ylabel('Number of samples')
plt.title(f'Non-IID {PARTITION_TYPE} Class Distribution (K={NUM_CLIENTS})')
plt.xticks(np.arange(NUM_CLIENTS))
plt.legend(ncol=10, loc='upper center', bbox_to_anchor=(0.5, -0.15))
plt.tight_layout()
partition_chart = PARTITION_DIR / f'{stem}_dist.png'
plt.savefig(partition_chart, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels=1, input_size=28):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(in_channels, 16, 5), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 5), nn.ReLU(), nn.MaxPool2d(2))
        feature_size = (input_size - 5 + 1) // 2
        feature_size = (feature_size - 5 + 1) // 2
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * feature_size * feature_size, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x):
        return self.classifier(self.features(x))

class LinearProbe(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.linear = nn.Linear(embedding_dim, 10)
    def forward(self, x):
        return self.linear(x)

class RawLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 10)
    def forward(self, x):
        return self.linear(x)

def embedding_cache_filename(split):
    key = f'{DATASET_NAME}_resnet18_imagenet1k_v1_resize{RESNET_SIZE}_imagenet_norm_{split}'
    return f'resnet18_sz{RESNET_SIZE}_{split}_{hashlib.sha256(key.encode()).hexdigest()[:8]}.pt'

def preprocess_resnet(images):
    x = images.unsqueeze(1).repeat(1, 3, 1, 1) if DATASET_NAME == 'mnist' else images
    x = F.interpolate(x, size=(RESNET_SIZE, RESNET_SIZE), mode='bilinear', align_corners=False)
    return (x - IMAGENET_MEAN) / IMAGENET_STD

def make_embeddings(images, backbone):
    backbone.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(images), EMBED_BATCH_SIZE):
            outputs.append(backbone(preprocess_resnet(images[start:start + EMBED_BATCH_SIZE].to(DEVICE))).float().cpu())
    return torch.cat(outputs)

if MODEL_NAME == 'resnet_linear':
    train_embed_path = DATA_DIR / embedding_cache_filename('train')
    test_embed_path = DATA_DIR / embedding_cache_filename('test')
    if train_embed_path.exists() and test_embed_path.exists():
        train_features = torch.load(train_embed_path, map_location='cpu')
        test_features = torch.load(test_embed_path, map_location='cpu')
    else:
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE)
        backbone.fc = nn.Identity()
        for parameter in backbone.parameters():
            parameter.requires_grad_(False)
        train_features = make_embeddings(train_images, backbone)
        test_features = make_embeddings(test_images, backbone)
        torch.save(train_features, train_embed_path)
        torch.save(test_features, test_embed_path)
        del backbone
        torch.cuda.empty_cache()
    train_features = train_features.to(DEVICE)
    test_features = test_features.to(DEVICE)

def build_model():
    if MODEL_NAME == 'cnn':
        return SmallCNN(INPUT_CHANNELS, INPUT_SIZE).to(DEVICE)
    if MODEL_NAME == 'logreg_raw':
        return RawLogisticRegression(INPUT_CHANNELS * INPUT_SIZE * INPUT_SIZE).to(DEVICE)
    return LinearProbe(512).to(DEVICE)

def batch_seed(round_id, client_id, local_epoch):
    return BATCH_SEED + round_id * 1_000_000 + client_id * 1_000 + local_epoch

def select_clients(round_id):
    if CLIENTS_PER_ROUND == NUM_CLIENTS:
        return list(range(NUM_CLIENTS))
    generator = torch.Generator(device='cpu').manual_seed(SELECTION_SEED + round_id)
    return torch.randperm(NUM_CLIENTS, generator=generator)[:CLIENTS_PER_ROUND].tolist()

def local_train(model, global_state, client_id, round_id):
    model.load_state_dict(global_state)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
    indices = client_indices[client_id]
    for local_epoch in range(LOCAL_EPOCHS):
        generator = torch.Generator(device='cpu').manual_seed(batch_seed(round_id, client_id, local_epoch))
        permutation = indices[torch.randperm(len(indices), generator=generator)].to(DEVICE)
        for start in range(0, len(permutation), BATCH_SIZE):
            batch_indices = permutation[start:start + BATCH_SIZE]
            loss = F.cross_entropy(model(train_features[batch_indices]), train_labels_gpu[batch_indices])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    return {name: parameter.detach().clone() for name, parameter in model.named_parameters()}, len(indices)

@torch.no_grad()
def flatten_state(state, names):
    return torch.cat([state[name].reshape(-1) for name in names])

def state_from_vector(vector, template, names):
    result, offset = {}, 0
    for name in names:
        size = template[name].numel()
        result[name] = vector[offset:offset + size].reshape(template[name].shape)
        offset += size
    return result

def choose_elbow_k(vectors, max_k, round_id):
    max_k = min(max_k, len(vectors))
    if max_k < 2:
        return 1, np.array([1]), np.array([0.0]), np.zeros(len(vectors), dtype=np.int64)
    ks = np.arange(2, max_k + 1)
    inertias, labels_by_k = [], {}
    for k in ks:
        model = KMeans(n_clusters=int(k), n_init=10, random_state=CLUSTER_SEED + round_id)
        model.fit(vectors)
        inertias.append(model.inertia_)
        labels_by_k[int(k)] = model.labels_.copy()
    points = np.column_stack([ks.astype(float), np.asarray(inertias)])
    normalized = (points - points.min(axis=0)) / np.maximum(points.max(axis=0) - points.min(axis=0), 1e-12)
    line = normalized[-1] - normalized[0]
    distances = np.abs(line[0] * (normalized[:, 1] - normalized[0, 1]) - line[1] * (normalized[:, 0] - normalized[0, 0])) / max(np.linalg.norm(line), 1e-12)
    selected_k = int(ks[int(np.argmax(distances))])
    return selected_k, ks, np.asarray(inertias), labels_by_k[selected_k]

@torch.no_grad()
def top1_cosine_aggregate(client_states, client_sizes, global_state, names, round_id):
    global_vector = flatten_state(global_state, names)
    update_vectors = torch.stack([flatten_state(state, names) - global_vector for state in client_states])
    selected_k, ks, inertias, cluster_labels = choose_elbow_k(update_vectors.cpu().numpy(), K_MAX, round_id)
    representative_indices = []
    for cluster_id in range(selected_k):
        members = np.flatnonzero(cluster_labels == cluster_id)
        member_updates = update_vectors[members]
        weights = torch.tensor([client_sizes[i] for i in members], dtype=member_updates.dtype, device=member_updates.device)
        weights = weights / weights.sum()
        centroid = (member_updates * weights[:, None]).sum(dim=0)
        similarities = F.cosine_similarity(member_updates, centroid.unsqueeze(0), dim=1, eps=1e-12)
        representative_indices.append(int(members[int(torch.argmax(similarities).item())]))
    selected_states = [client_states[i] for i in representative_indices]
    selected_sizes = [client_sizes[i] for i in representative_indices]
    total = float(sum(selected_sizes))
    aggregated = {name: sum(state[name] * (size / total) for state, size in zip(selected_states, selected_sizes)) for name in names}
    return aggregated, selected_k, ks, inertias, cluster_labels, representative_indices

@torch.no_grad()
def evaluate(model, features, labels):
    model.eval()
    total_loss, total_correct = 0.0, 0
    for start in range(0, len(labels), EVAL_BATCH_SIZE):
        logits = model(features[start:start + EVAL_BATCH_SIZE])
        batch_labels = labels[start:start + EVAL_BATCH_SIZE]
        total_loss += F.cross_entropy(logits, batch_labels, reduction='sum').item()
        total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
    return total_loss / len(labels), total_correct / len(labels)

In [ ]:
METRIC_FIELDS = ['round', 'global_train_loss', 'global_train_accuracy', 'test_loss', 'test_accuracy', 'learning_rate', 'num_clients', 'num_selected_clients', 'selected_clients', 'num_samples', 'selected_k']

def load_metrics():
    if not METRICS_PATH.exists():
        return []
    with METRICS_PATH.open(newline='') as handle:
        return list(csv.DictReader(handle))

def truncate_metrics(round_id):
    rows = [row for row in load_metrics() if int(row['round']) <= round_id]
    tmp_path = METRICS_PATH.with_suffix('.tmp')
    with tmp_path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
        writer.writeheader()
        writer.writerows([{name: row.get(name, '') for name in METRIC_FIELDS} for row in rows])
    os.replace(tmp_path, METRICS_PATH)

def append_metric(row):
    if METRICS_PATH.exists():
        with METRICS_PATH.open(newline='') as handle:
            reader = csv.DictReader(handle)
            old_rows, old_fields = list(reader), reader.fieldnames or []
        if old_fields != METRIC_FIELDS:
            tmp_path = METRICS_PATH.with_suffix('.schema.tmp')
            with tmp_path.open('w', newline='') as handle:
                writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
                writer.writeheader()
                writer.writerows([{name: old_row.get(name, '') for name in METRIC_FIELDS} for old_row in old_rows])
            os.replace(tmp_path, METRICS_PATH)
    exists = METRICS_PATH.exists()
    with METRICS_PATH.open('a', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow(row)
        handle.flush()
        os.fsync(handle.fileno())

def save_checkpoint(round_id, model, selected_clients):
    checkpoint = {'round': round_id, 'model_state': {name: value.detach().cpu() for name, value in model.named_parameters()}, 'dirichlet_alpha': DIRICHLET_ALPHA, 'partition_type': PARTITION_TYPE, 'partition_seed': PARTITION_SEED, 'model_seed': MODEL_SEED, 'batch_seed': BATCH_SEED, 'selection_seed': SELECTION_SEED, 'cluster_seed': CLUSTER_SEED, 'k_max': K_MAX, 'selected_clients': selected_clients, 'torch_rng_state': torch.get_rng_state(), 'cuda_rng_state': torch.cuda.get_rng_state_all()}
    tmp_path = CHECKPOINT_PATH.with_suffix('.pt.tmp')
    torch.save(checkpoint, tmp_path)
    os.replace(tmp_path, CHECKPOINT_PATH)

current_config = {'model_name': MODEL_NAME, 'dataset_name': DATASET_NAME, 'num_clients': NUM_CLIENTS, 'clients_per_round': CLIENTS_PER_ROUND, 'local_epochs': LOCAL_EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE, 'resnet_size': RESNET_SIZE, 'dirichlet_alpha': DIRICHLET_ALPHA, 'partition_type': PARTITION_TYPE, 'partition_seed': PARTITION_SEED, 'model_seed': MODEL_SEED, 'batch_seed': BATCH_SEED, 'selection_seed': SELECTION_SEED, 'cluster_seed': CLUSTER_SEED, 'k_max': K_MAX}
seed_everything(MODEL_SEED)
global_model = build_model()
start_round = 1
if CHECKPOINT_PATH.exists():
    if CONFIG_PATH.exists():
        with CONFIG_PATH.open() as handle:
            saved_config = json.load(handle)
        mismatches = [f'  - {key}: saved={saved_config[key]}, current={value}' for key, value in current_config.items() if key in saved_config and saved_config[key] != value]
        if mismatches:
            raise ValueError(f'Run {RUN_ID} configuration changed:\n' + '\n'.join(mismatches) + '\nUse a new RUN_ID.')
        saved_config['rounds'] = ROUNDS
        with CONFIG_PATH.open('w') as handle:
            json.dump(saved_config, handle, indent=2)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
    global_model.load_state_dict(checkpoint['model_state'], strict=True)
    torch.set_rng_state(checkpoint['torch_rng_state'])
    torch.cuda.set_rng_state_all(checkpoint['cuda_rng_state'])
    completed_round = checkpoint['round']
    truncate_metrics(completed_round)
    start_round = completed_round + 1
    print(f'Resuming Run {RUN_ID} from round {start_round} to {ROUNDS}.')
else:
    with CONFIG_PATH.open('w') as handle:
        json.dump({'run_id': RUN_ID, 'run_dir': str(RUN_DIR), 'rounds': ROUNDS, **current_config, 'partition_path': str(partition_path), 'device': str(DEVICE)}, handle, indent=2)
    print(f'Starting Run {RUN_ID} from round 1 to {ROUNDS}.')

if CLUSTER_LOG_PATH.exists():
    with CLUSTER_LOG_PATH.open('r') as handle:
        cluster_log = json.load(handle)
else:
    cluster_log = {}

worker_model = build_model()
names = [name for name, _ in global_model.named_parameters()]
for round_id in range(start_round, ROUNDS + 1):
    round_start = time.time()
    selected_clients = select_clients(round_id)
    global_state = {name: parameter.detach().clone() for name, parameter in global_model.named_parameters()}
    client_states, client_sizes = [], []
    for client_id in selected_clients:
        state, size = local_train(worker_model, global_state, client_id, round_id)
        client_states.append(state)
        client_sizes.append(size)
    new_state, selected_k, ks, inertias, cluster_labels, representative_indices = top1_cosine_aggregate(client_states, client_sizes, global_state, names, round_id)
    representative_clients = [selected_clients[i] for i in representative_indices]
    cluster_log[str(round_id)] = {
        'client_clusters': {
            str(client_id): int(label) + 1
            for client_id, label in zip(selected_clients, cluster_labels)
        },
        'representative_clients': {
            str(cluster_id + 1): client_id
            for cluster_id, client_id in enumerate(representative_clients)
        },
    }
    cluster_log_tmp_path = CLUSTER_LOG_PATH.with_suffix('.tmp')
    with cluster_log_tmp_path.open('w') as handle:
        json.dump(cluster_log, handle, indent=2, sort_keys=True)
    os.replace(cluster_log_tmp_path, CLUSTER_LOG_PATH)
    global_model.load_state_dict(new_state, strict=True)
    global_train_loss, global_train_accuracy = evaluate(global_model, train_features, train_labels_gpu)
    test_loss, test_accuracy = evaluate(global_model, test_features, test_labels_gpu)
    append_metric({'round': round_id, 'global_train_loss': global_train_loss, 'global_train_accuracy': global_train_accuracy, 'test_loss': test_loss, 'test_accuracy': test_accuracy, 'learning_rate': LEARNING_RATE, 'num_clients': NUM_CLIENTS, 'num_selected_clients': len(selected_clients), 'selected_clients': json.dumps(selected_clients), 'num_samples': sum(client_sizes), 'selected_k': selected_k})
    save_checkpoint(round_id, global_model, selected_clients)
    print(f'Round {round_id:02d}/{ROUNDS} | K={selected_k} | clients {selected_clients} | test acc {test_accuracy:.2%} | {time.time() - round_start:.1f}s')

del worker_model
torch.save({name: value.detach().cpu() for name, value in global_model.named_parameters()}, RUN_DIR / 'final_model_parameters.pt')
print('Finished. Results:', RUN_DIR)

In [ ]:
# Load and plot a completed or partially completed run
plot_run_id = RUN_ID
plot_run_dir = RUN_DIR if plot_run_id == RUN_ID else EXP_DIR / (str(plot_run_id) if str(plot_run_id).startswith('run') else f'run{plot_run_id}')
plot_metrics_path = plot_run_dir / 'metrics.csv'
if not plot_metrics_path.exists():
    print(f'No metrics found at {plot_metrics_path}')
else:
    with plot_metrics_path.open(newline='') as handle:
        rows = list(csv.DictReader(handle))
    if rows:
        rounds = [int(row['round']) for row in rows]
        accuracy = [float(row['test_accuracy']) for row in rows]
        train_accuracy = [float(row['global_train_accuracy']) for row in rows]
        train_loss = [float(row['global_train_loss']) for row in rows]
        test_loss = [float(row['test_loss']) for row in rows]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(rounds, accuracy, marker='o', label='Test accuracy')
        axes[0].plot(rounds, train_accuracy, marker='x', linestyle='--', label='Train accuracy')
        axes[0].set(xlabel='FedAvg round', ylabel='Accuracy', title='Top-1 Cosine Clustered FedAvg')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        axes[1].plot(rounds, train_loss, label='Global train loss')
        axes[1].plot(rounds, test_loss, label='Test loss')
        axes[1].set(xlabel='FedAvg round', ylabel='Loss', title='Global Losses')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        result_chart = plot_run_dir / 'metrics.png'
        plt.savefig(result_chart, dpi=150)
        plt.show()
        summary = {'run_id': plot_run_id, 'run_dir': str(plot_run_dir), 'model_name': MODEL_NAME, 'dirichlet_alpha': DIRICHLET_ALPHA, 'completed_rounds': len(rounds), 'final_test_accuracy': accuracy[-1], 'best_test_accuracy': max(accuracy), 'final_train_loss': train_loss[-1], 'final_test_loss': test_loss[-1], 'final_train_accuracy': train_accuracy[-1]}
        with (plot_run_dir / 'summary.json').open('w') as handle:
            json.dump(summary, handle, indent=2)
        print('Final test accuracy:', accuracy[-1])